In [ ]:
# import necessary packages
import pandas as pd
import re
from rdkit import Chem
from rdkit.Chem import Descriptors, Fragments, AllChem, Draw
from rdkit.Chem.Draw import rdMolDraw2D
from collections import Counter
from IPython.display import SVG, display
import xml.etree.ElementTree as ET

The xml file containting the Drugbank database was downloaded to the same folder as this notebook from https://go.drugbank.com/releases/latest, with the appropriate academic license.

In [ ]:
# define the function to extract SMILES from the xml file
def extract_smiles(xml_file):
    with open(xml_file) as f:
        text = f.read()

    pattern = r"<kind>SMILES</kind>\s*<value>(.*?)</value>"
    return re.findall(pattern, text, re.DOTALL)

In [ ]:
# extract the SMILES of compounds in the drugbank database
drugbank_database = 'full database.xml'
smiles = extract_smiles(drugbank_database)
# for s in smiles:
    # print(s.strip())

smiles_list = smiles

In [ ]:
# define functions to process the data
def find_aryl_substituents_with_attachment(smiles):
    """
    Identify aromatic rings and extract substituents with their attachment points.
    Returns list of (substituent_mol, attachment_atom_idx_in_substituent)
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return []
    
    substituents = []
    
    # Find aromatic rings
    ri = mol.GetRingInfo()
    aromatic_atoms = set()
    
    for ring in ri.AtomRings():
        if len(ring) == 6:  # Focus on 6-membered rings (benzene-like)
            atoms_in_ring = [mol.GetAtomWithIdx(i) for i in ring]
            if all(atom.GetIsAromatic() for atom in atoms_in_ring):
                aromatic_atoms.update(ring)
    
    # For each aromatic atom, check for substituents
    for ar_idx in aromatic_atoms:
        ar_atom = mol.GetAtomWithIdx(ar_idx)
        
        for neighbor in ar_atom.GetNeighbors():
            neighbor_idx = neighbor.GetIdx()
            
            # If neighbor is not in the aromatic ring, it's a substituent
            if neighbor_idx not in aromatic_atoms:
                try:
                    # Use RDKit's fragmentation with bond breaking
                    # First, mark the bond to break
                    bond_idx = mol.GetBondBetweenAtoms(ar_idx, neighbor_idx).GetIdx()
                    
                    # Fragment the molecule at this bond
                    fragmented = Chem.FragmentOnBonds(mol, [bond_idx], addDummies=False)
                    
                    # Get the fragments
                    frags = Chem.GetMolFrags(fragmented, asMols=True, sanitizeFrags=True)
                    
                    # Find the substituent fragment (the one NOT containing the aromatic core)
                    for frag in frags:
                        # Check if this fragment has fewer atoms and doesn't contain most of the aromatic atoms
                        frag_atom_indices = set()
                        
                        # We need to identify which fragment is the substituent
                        # The substituent should be smaller and not contain the aromatic core
                        if frag.GetNumAtoms() < mol.GetNumAtoms() / 2:  # Heuristic: substituent is smaller
                            # Now find the attachment point in the fragment
                            # The attachment point is the atom that was bonded to the aromatic ring
                            
                            # We need to track which atom in the fragment was connected
                            # Use a different approach: break bond and add dummy atom
                            mol_with_dummy = Chem.FragmentOnBonds(mol, [bond_idx], addDummies=True)
                            frags_with_dummy = Chem.GetMolFrags(mol_with_dummy, asMols=True, sanitizeFrags=True)
                            
                            for frag_dummy in frags_with_dummy:
                                # Find the fragment with the dummy atom (*)
                                has_dummy = False
                                dummy_idx = None
                                
                                for atom_idx, atom in enumerate(frag_dummy.GetAtoms()):
                                    if atom.GetAtomicNum() == 0:  # Dummy atom
                                        has_dummy = True
                                        dummy_idx = atom_idx
                                        break
                                
                                if has_dummy and frag_dummy.GetNumAtoms() <= mol.GetNumAtoms() / 2:
                                    # This is the substituent with dummy atom marking attachment
                                    # The atom connected to the dummy is the attachment point
                                    dummy_atom = frag_dummy.GetAtomWithIdx(dummy_idx)
                                    attachment_neighbors = dummy_atom.GetNeighbors()
                                    
                                    if len(attachment_neighbors) == 1:
                                        attachment_idx = attachment_neighbors[0].GetIdx()
                                        
                                        # Remove the dummy atom to get clean substituent
                                        editable = Chem.EditableMol(frag_dummy)
                                        editable.RemoveAtom(dummy_idx)
                                        clean_frag = editable.GetMol()
                                        
                                        try:
                                            Chem.SanitizeMol(clean_frag)
                                            # Adjust attachment index if dummy was before it
                                            if dummy_idx < attachment_idx:
                                                attachment_idx -= 1
                                            
                                            substituents.append((clean_frag, attachment_idx))
                                        except:
                                            pass
                except Exception as e:
                    continue
    
    return substituents

def attach_to_astatine(substituent_mol, attachment_idx):
    """
    Attach the substituent to astatine at the specified attachment point.
    """
    if substituent_mol is None:
        return None
    
    try:
        # Create astatine atom
        at_mol = Chem.MolFromSmiles('[At]')
        
        # Combine molecules
        combined = Chem.CombineMols(at_mol, substituent_mol)
        
        # Create editable molecule
        editable = Chem.EditableMol(combined)
        
        # Add bond between At (index 0) and attachment point (index 1 + attachment_idx)
        # Index 0 is At, indices 1+ are from the substituent
        editable.AddBond(0, 1 + attachment_idx, Chem.BondType.SINGLE)
        
        final_mol = editable.GetMol()
        Chem.SanitizeMol(final_mol)
        
        return Chem.MolToSmiles(final_mol)
    except Exception as e:
        return None

def count_rotatable_bonds(smiles):
    """Count the number of rotatable bonds in a molecule"""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return Descriptors.NumRotatableBonds(mol)

In [ ]:
# do the above on the identified substituents
def main():
    
    all_substituents = []
    
    # find the aryl substituents
    for i, smiles in enumerate(smiles_list):
        substituents = find_aryl_substituents_with_attachment(smiles)
        all_substituents.extend(substituents)
    
    at_substituents = []
    
    # attach the substituents to astatine
    for sub_mol, attach_idx in all_substituents:
        at_sub = attach_to_astatine(sub_mol, attach_idx)
        if at_sub:
            at_substituents.append(at_sub)
    
    # Count frequency of each substituent
    frequency_counter = Counter(at_substituents)
    
    # filter substituents by rotatable bonds
    filtered_substituents = []
    
    for at_sub, frequency in frequency_counter.items():
        n_rot = count_rotatable_bonds(at_sub)
        if n_rot is not None and n_rot <= 2:
            filtered_substituents.append({
                'smiles': at_sub,
                'rotatable_bonds': n_rot,
                'frequency': frequency
            })
    
    # save the substituents to a csv
    df = pd.DataFrame(filtered_substituents)
    df = df.sort_values('frequency', ascending=False)  # Sort by most common first
    df.to_csv('aryl_substituents_astatine.csv', index=False)

if __name__ == "__main__":
    main()

In [ ]:
# display the dataset
df_drugbank = pd.read_csv('aryl_substituents_astatine.csv')

df_drugbank.head()

In [ ]:
# remove substituents that do not occur at least 15 times
df_drugbank = df_drugbank[df_drugbank['frequency'] >= 15]

len(df_drugbank)

In [ ]:
# Filter out drugbank substituents with formal charges
def has_formal_charge(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return False
    for atom in mol.GetAtoms():
        if atom.GetFormalCharge() != 0:
            return True
    return False

df_drugbank = df_drugbank[~df_drugbank['smiles'].apply(has_formal_charge)]

len(df_drugbank)

In [ ]:
# define acidic substructures
# smarts determined via university of hamburg's SMARTS Editor
acid1 = Chem.MolFromSmarts('[*X3](=O)[OX2H1]') #carboxylic acid and analogs
acid2 = Chem.MolFromSmarts('P(*)(O)(=O)O') #phosphonic acid
acid3 = Chem.MolFromSmarts('S(*)(O)(=O)=O') #sulfonic acid
acids = [acid1, acid2, acid3]

In [ ]:
# create mol objects for the drugbank substituents
mol_objects = [Chem.MolFromSmiles(s) for s in df_drugbank['smiles']]

df_drugbank['mol'] = mol_objects

In [ ]:
# Remove acidic substructures from the drugbank database
matching_mols = []
for acid in acids:
    matching_mols.extend([mol for mol in mol_objects if mol.HasSubstructMatch(acid)])

df_drugbank = df_drugbank[~df_drugbank['mol'].isin(matching_mols)]

len(df_drugbank)

In [ ]:
# read in my dataset
my_dataset = 'hammett_predictions.csv'

df_me = pd.read_csv(my_dataset)
df_me = df_me.drop(columns=['cm5_charge','hammett_parameter','predicted_hammett','nitrile_smiles'])

df_me.head()

In [ ]:
# canonicalize smiles to ensure that we are correctly comparing substituents
# Canonicalize SMILES in df_me['Substituent']
df_me['Substituent'] = df_me['Substituent'].apply(lambda x: Chem.MolToSmiles(Chem.MolFromSmiles(x), canonical=True) if Chem.MolFromSmiles(x) else x)

# Canonicalize SMILES in df_drugbank['smiles']
df_drugbank['smiles'] = df_drugbank['smiles'].apply(lambda x: Chem.MolToSmiles(Chem.MolFromSmiles(x), canonical=True) if Chem.MolFromSmiles(x) else x)

In [ ]:
# identify those substituents which are not present in my dataset
df_not_in_me = df_drugbank[~df_drugbank['smiles'].isin(df_me['Substituent'])]

len(df_not_in_me)

In [ ]:
# visualize the substituents
Draw.MolsToGridImage([Chem.MolFromSmiles(s) for s in df_not_in_me['smiles']], molsPerRow=8, subImgSize=(200,200),maxMols=len(df_not_in_me))

In [ ]:
# remove the specific isotope substituent
df_not_in_me = df_not_in_me[~df_not_in_me['smiles'].str.contains(r'\[123I\]\[At\]', regex=True)]

In [ ]:
len(df_not_in_me)

In [ ]:
# create a column for mol objects for the identified substituents
df_not_in_me['mols'] = df_not_in_me['smiles'].apply(lambda x: Chem.MolFromSmiles(x))

In [ ]:
# define benzonitrile to react with the substituents
benzonitrile = Chem.MolFromSmiles('N#CC1=CC=CC=C1')

# Define the SMARTS reaction to make the para-substituted benzonitriles. The substituents were attached to At as a dummy atom. 
para_rxn = AllChem.ReactionFromSmarts('[N:1]#[C:2]-[c,C:3]1[c,C:4][c,C:5][c,C:6][c,C:7][c,C:8]1.[At]-[*:9]>>[N:1]#[C:2]-[c:3]1[c:4][c:5][c:6](-[*:9])[c:7][c:8]1')
para_rxn

In [ ]:
# Define the SMARTS reaction to make the meta-substituted benzonitriles.
meta_rxn = AllChem.ReactionFromSmarts('[N:1]#[C:2]-[c,C:3]1[c,C:4][c,C:5][c,C:6][c,C:7][c,C:8]1.[At]-[*:9]>>[N:1]#[C:2]-[c:3]1[c:4][c:5](-[*:9])[c:6][c:7][c:8]1')
meta_rxn

In [ ]:
# Run the SMARTS reaction to create the para- and meta-substituted benzonitriles for all of the substituents, then convert the products into SMILES strings.
para_products = []
for subst in df_not_in_me['mols']:
    print(f"Reacting benzonitrile with {subst}")
    result = para_rxn.RunReactants((benzonitrile, subst))
    print(f"Result: {result}")
    if result and result[0]:  # Check if there is at least one product
        para_products.append(result[0][0])
    else:
        para_products.append(None)  

para_smis = [Chem.MolToSmiles(mol) if mol is not None else '' for mol in para_products]
df_not_in_me['para_nitrile'] = para_smis

meta_products = []
for subst in df_not_in_me['mols']:
    print(f"Reacting benzonitrile with {subst}")
    result = meta_rxn.RunReactants((benzonitrile, subst))
    print(f"Result: {result}")
    if result and result[0]:  # Check if there is at least one product
        meta_products.append(result[0][0])
    else:
        meta_products.append(None)  

meta_smis = [Chem.MolToSmiles(mol) if mol is not None else '' for mol in meta_products]
df_not_in_me['meta_nitrile'] = meta_smis

In [ ]:
# OPTIONAL – visualize the para-substituted benzonitriles
Draw.MolsToGridImage([Chem.MolFromSmiles(s) for s in df_not_in_me['para_nitrile']], len(df_not_in_me), molsPerRow=8, subImgSize=(200,200))

In [ ]:
# create unique ids for para- and meta-substituted benzonitriles
para_nitrile_names = ['para_nitrile_n{:02d}'.format(i) for i in range(len(para_smis))]
meta_nitrile_names = ['meta_nitrile_n{:02d}'.format(i) for i in range(len(meta_smis))]
df_not_in_me['para_nitrile_id'] = para_nitrile_names
df_not_in_me['meta_nitrile_id'] = meta_nitrile_names

In [ ]:
# Save the filtered DataFrame with para- and meta-substituted benzonitriles to CSV files
df_not_in_me.drop(columns=['mols']).to_csv('new_nitriles.csv')
df_not_in_me[['para_nitrile_id', 'para_nitrile']].to_csv('new_para_nitriles.csv')
df_not_in_me[['meta_nitrile_id', 'meta_nitrile']].to_csv('new_meta_nitriles.csv')

# rename columns so the csvs can be used to generate 3D conformers with AQME in the next step of the workflow
para=pd.read_csv('new_para_nitriles.csv')
para = para.rename(columns={'para_nitrile_id': 'code_name', 'para_nitrile': 'SMILES'})
para.to_csv('new_para_nitriles.csv', index=False)

meta=pd.read_csv('new_meta_nitriles.csv')
meta = meta.rename(columns={'meta_nitrile_id': 'code_name', 'meta_nitrile': 'SMILES'})
meta.to_csv('new_meta_nitriles.csv', index=False)

In [ ]:
# use AQME to generate 3D conformers from the SMILES of the para-substituted benzonitriles
! python -m aqme --csearch --program rdkit --input new_para_nitriles.csv 

In [ ]:
! python -m aqme --csearch --program rdkit --input new_meta_nitriles.csv

In [ ]:
# use AQME to prepare the conformers for quantum mechanical optimization, using the benchmarked level of theory
# Here we use Gaussian to perform the calculations, but the program can be changed based on your needs
! python -m aqme --qprep --program gaussian --qm_input "B3LYP def2SVP emp=GD3BJ opt freq" --files "CSEARCH/*.sdf"
# OPTIONAL – rename the QCALC directory
! mv QCALC optimization_inputs

In [ ]:
# use AQME to check the optimization output files for successful completion
# AQME will generate new input files for those calculations that did not terminate successfully/did not optimize to a minimum. Resubmit these calculations.
! python -m aqme --qcorr --program gaussian --files "optimization_outputs/*.log"

In [ ]:
# Use AQME to generate single point calculation inputs for the successfully optimized conformers, using the benchmarked level of theory
# pop=(Hirshfeld) will compute both Hirshfeld and CM5 charges that will be used for predictions
! python -m aqme --qprep --program gaussian --qm_input "B3LYP def2QZVPP emp=GD3BJ pop=(Hirshfeld)" --files "optimization_outputs/success/*.log"
# OPTIONAL – rename the directory containing the input files
! mv QCALC single_point_inputs

In [ ]:
# Use AQME to check the single point output files for successful completion
! python -m aqme --qcorr --program gaussian --files "single_point_outputs/*.log"

In [ ]:
# OPTIONAL – rename the directories for the molDscript input and separate meta and para – substituted benzonitriles
! mkdir opt 
! cp optimization_outputs/success/*.log opt
! mkdir spc 
! cp single_point_outputs/success/SP_calcs/*.log spc

In [ ]:
# substructure obtained using University of Hamburg SMARTSeditor

# use molDscript to parse atomic charges from the meta single point calculations
! python -m moldscript --opt opt --spc spc --charges spc --substructure 'cC#N' --boltz --min_max --lowe
! mkdir new_charges
! mv *_level.csv new_charges
! mv MOLDSCRIPT*.dat new_charges

In [ ]:
# Organize the molDscript output data. The atom_level csv contains the atomic charge information.
dir = 'new_charges/'
df = pd.read_csv(dir + 'lowest_energy_atom_level.csv')

# separate N from other atom types
df_N = df[df['atom_type'] == 'N']

# remove the 'rdkit' suffix
df_N['filename'] = df_N['filename'].apply(lambda x: '_'.join([x.split('_')[0], x.split('_')[1], x.split('_')[2].zfill(3)]))

# reorder the dataframe
df_N = df_N.sort_values(by='filename')

# rename the columns to match the expected format
df_N = df_N.rename(columns={'filename': 'nitrile_id'})

In [ ]:
# identify any benzonitriles with more than one N atom (incorrectly parsed charges)
repeats = df_N[df_N.duplicated('nitrile_id', keep=False)]
repeats_df = repeats.copy()
print(repeats_df)

In [ ]:
# save the filtered charges to a new csv file
df_N.to_csv('new_charges.csv', index=False)

In [ ]:
# read in the new data
new_charges = 'new_charges.csv'

df = pd.read_csv(new_charges)

In [ ]:
# predict Hammett parameters for the new substituents
df['predicted_hammett'] = 34.61 * df['cm5_charge'] + 11.93

In [ ]:
# separate the para and meta predictions for each nitrile id
def restructure_hammett_data(df):
    """
    Restructure dataframe to have separate columns for meta and para predictions.
    
    Args:
        df: DataFrame with columns 'nitrile_id' and 'predicted_hammett'
        
    Returns:
        DataFrame with columns 'nitrile', 'predicted_sigma_m', 'predicted_sigma_p'
    """
    # Extract the n## part from nitrile_id
    df['nitrile'] = df['nitrile_id'].str.extract(r'(n\d+)')[0]
    
    # Create separate dataframes for meta and para
    meta_df = df[df['nitrile_id'].str.startswith('meta')][['nitrile', 'predicted_hammett']].copy()
    meta_df = meta_df.rename(columns={'predicted_hammett': 'predicted_sigma_m'})
    
    para_df = df[df['nitrile_id'].str.startswith('para')][['nitrile', 'predicted_hammett']].copy()
    para_df = para_df.rename(columns={'predicted_hammett': 'predicted_sigma_p'})
    
    # Merge on nitrile
    result = pd.merge(meta_df, para_df, on='nitrile', how='outer')
    
    return result

new_df = restructure_hammett_data(df)

In [ ]:
new_df.head()

In [ ]:
# add the SMILES for each substituent to the dataframe
smiles = 'new_nitriles.csv'
smiles_df = pd.read_csv(smiles)

smiles_df['nitrile'] = smiles_df['para_nitrile_id'].str.extract(r'(n\d+)')[0]
smiles_df = smiles_df.drop(columns=['para_nitrile', 'meta_nitrile', 'meta_nitrile_id','para_nitrile_id','rotatable_bonds','Unnamed: 0'])

In [ ]:
# combine the SMILES and new predictions in one dataframe
predictions = pd.merge(new_df, smiles_df, on='nitrile', how='left')

# save the new predictions to a csv file
predictions.to_csv('new_predictions.csv', index=False)

In [ ]:
df_fig = pd.read_csv('new_predictions.csv')

In [ ]:
# OPTIONAL – draw the substituents with their predicted Hammett parameters
df_fig['smiles'] = df_fig['smiles'].str.replace('[At]', '[*]', regex=False)

smiles_list = df_fig['smiles'].tolist()
sigma_p = df_fig['predicted_sigma_p'].tolist()
sigma_m = df_fig['predicted_sigma_m'].tolist()
output_svg = 'new_predictions_correct_sigmas.svg'

def draw_molecules_bw(smiles_list, sigma_p, sigma_m, output_svg, mols_per_row=7, mol_size=(300, 300)):
    """Draw molecules in black and white with dummies as attachments and sigma values as legend."""
    
    # Filter valid molecules and keep corresponding sigma values
    valid_data = [(Chem.MolFromSmiles(smi), p, m) 
                  for smi, p, m in zip(smiles_list, sigma_p, sigma_m) 
                  if Chem.MolFromSmiles(smi) is not None]
    
    mols = [mol for mol, p, m in valid_data]
    legends = [f"p = {p:.2f}\nm = {m:.2f}" for mol, p, m in valid_data]
    
    drawer = rdMolDraw2D.MolDrawOptions()
    drawer.useBWAtomPalette()
    drawer.dummiesAreAttachments = True
    drawer.padding = 0.05
    
    # Draw the molecules
    svg = Draw.MolsToGridImage(
        mols, 
        molsPerRow=mols_per_row, 
        subImgSize=mol_size, 
        legends=legends,
        drawOptions=drawer,
        useSVG=True,
        maxMols=len(df_fig)
    )
    
    # Convert SVG object to string
    svg_str = svg.data
    
    # Save SVG to file
    with open(output_svg, 'w') as f:
        f.write(svg_str)
    
    # Display in Jupyter notebook
    display(svg)
    
    return svg
    
# Usage:
img = draw_molecules_bw(smiles_list, sigma_p, sigma_m, output_svg)